 A detecção e o rastreamento dos sistemas convectivos foram realizados localmente em ambiente Jupyter Notebook, onde o TATHU apresentou execução estável. Essa etapa gera o arquivo `dataset_mlp_regressao_deslocamento.csv`. e as figuras salvas no drive e github

In [ ]:
#importação
import glob
import os
import warnings
import pandas as pd
import numpy as np
import copy
import torch

from osgeo import gdal
from shapely.errors import ShapelyDeprecationWarning

from tathu.constants import LAT_LONG_WGS84_BRAZIL_NORTH_EXTENT
from tathu.satellite import goes13
from tathu.tracking import detectors, descriptors, trackers, forecasters
from tathu.tracking.utils import area2degrees
from tathu.utils import file2timestamp
from tathu.visualizer import MapView

warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

In [ ]:
#define  domínio, resolução, limiar e pasta dos dados

# Pasta onde estão os arquivos GOES -
# caminho do drive:
#"/content/drive/MyDrive/Doutorado/Neurocomp/artigo final/load_data/GOES13_CH4_JUN2011_BIN"
data_dir = "/mnt/e/GOES12/jun2011/ch4"


# ── Domínio e detecção ────────────────────────────────────────────────────────
EXTENT     = LAT_LONG_WGS84_BRAZIL_NORTH_EXTENT
RESOLUTION = 4.0          # km
THRESHOLD  = 230          # K  (Tb mínima)
MINAREA    = area2degrees(3000)  # km² → graus²
OVERLAP    = 0.1          # critério de sobreposição relativa para o tracker


In [ ]:
# Leitura e Inspeção dos Arquivos GOES-13

query = os.path.join(data_dir, "06_*", "S10216956_*")
files = sorted([f for f in glob.glob(query) if os.path.isfile(f)])

print(f"Total de arquivos: {len(files)}")
print("Primeiros:", files[:3])
print("Últimos:  ", files[-3:])

# Constrói tabela de arquivos com timestamps
times = [
    file2timestamp(f, regex=goes13.DATE_REGEX, format=goes13.DATE_FORMAT)
    for f in files
]

df_files = (
    pd.DataFrame({"file": files, "timestamp": times})
    .sort_values("timestamp")
    .reset_index(drop=True)
)
df_files["dt_horas"] = df_files["timestamp"].diff().dt.total_seconds() / 3600

print("\nIntervalos entre arquivos (horas):")
print(df_files["dt_horas"].value_counts().sort_index())
print(df_files.head())

Total de arquivos: 477
Primeiros: ['/mnt/e/GOES12/jun2011/ch4/06_07/S10216956_201106070000', '/mnt/e/GOES12/jun2011/ch4/06_07/S10216956_201106070100', '/mnt/e/GOES12/jun2011/ch4/06_07/S10216956_201106070200']
Últimos:   ['/mnt/e/GOES12/jun2011/ch4/06_29/S10216956_201106291800', '/mnt/e/GOES12/jun2011/ch4/06_29/S10216956_201106292000', '/mnt/e/GOES12/jun2011/ch4/06_29/S10216956_201106292100']

Intervalos entre arquivos (horas):
dt_horas
1.0    426
2.0     27
3.0     23
Name: count, dtype: int64
                                                file           timestamp  \
0  /mnt/e/GOES12/jun2011/ch4/06_07/S10216956_2011... 2011-06-07 00:00:00   
1  /mnt/e/GOES12/jun2011/ch4/06_07/S10216956_2011... 2011-06-07 01:00:00   
2  /mnt/e/GOES12/jun2011/ch4/06_07/S10216956_2011... 2011-06-07 02:00:00   
3  /mnt/e/GOES12/jun2011/ch4/06_07/S10216956_2011... 2011-06-07 03:00:00   
4  /mnt/e/GOES12/jun2011/ch4/06_07/S10216956_2011... 2011-06-07 04:00:00   

   dt_horas  
0       NaN  
1       1.0  
2 

In [ ]:
#  Função de Detecção e Descrição

def detect(path, grids=None):
    """
    Lê um arquivo GOES-13, detecta SCMs com Tb < THRESHOLD e
    calcula atributos estatísticos.

    Parameters
    ----------
    path   : caminho do arquivo
    grids  : lista opcional; se fornecida, o grid remapeado é adicionado

    Returns
    -------
    list de sistemas detectados com timestamp e atributos preenchidos
    """
    timestamp = file2timestamp(
        path, regex=goes13.DATE_REGEX, format=goes13.DATE_FORMAT
    ).replace(microsecond=0)

    print(f"  Processing {timestamp}")

    grid = goes13.sat2grid(
        path, EXTENT, RESOLUTION, progress=gdal.TermProgress_nocb
    )

    systems = detectors.LessThan(THRESHOLD, MINAREA).detect(grid)
    print(f"  Sistemas detectados: {len(systems)}")

    for s in systems:
        s.timestamp = timestamp

    descriptor = descriptors.StatisticalDescriptor(rasterOut=True)
    descriptor.describe(grid, systems)

    for s in systems:
        s.attrs["nae"] = 0

    if grids is not None:
        grids.append(grid)

    return systems


# Teste rápido com o primeiro arquivo
print("=== Teste de detecção ===")
sistemas_teste = detect(files[0])
print(f"Atributos do primeiro sistema: {sistemas_teste[0].attrs}")

=== Teste de detecção ===
  Processing 2011-06-07 00:00:00
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 11
Atributos do primeiro sistema: {'touching_left': False, 'touching_right': False, 'touching_up': True, 'touching_down': False, 'min': 208.5800018310547, 'mean': 220.92742026055706, 'count': 1113, 'std': 5.580595533484621, 'nae': 0}


Warning 1: DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.
/home/sialm/miniforge3/envs/env-tathu/lib/python3.13/site-packages/rasterstats/io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


In [ ]:
# Rastreamento Temporal Completo

strategy = trackers.RelativeOverlapAreaStrategy(OVERLAP)

previous = None
all_systems_tracked = []

for i, path in enumerate(files, start=1):
    print(f"\n[{i}/{len(files)}] {os.path.basename(path)}")

    try:
        current = detect(path)

        if previous is not None:
            tracker = trackers.OverlapAreaTracker(previous, strategy=strategy)
            tracker.track(current)

        all_systems_tracked.extend(current)
        previous = current

    except Exception as exc:
        print(f"  ERRO em {path}: {exc}")

print(f"\nTotal de sistemas rastreados: {len(all_systems_tracked)}")


[1/477] S10216956_201106070000
  Processing 2011-06-07 00:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 11

[2/477] S10216956_201106070100
  Processing 2011-06-07 01:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 9

[3/477] S10216956_201106070200
  Processing 2011-06-07 02:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 7

[4/477] S10216956_201106070300
  Processing 2011-06-07 03:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 6

[5/477] S10216956_201106070400
  Processing 2011-06-07 04:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 7

[6/477] S10216956_201106070500
  Processing 2011-06-07 05:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 6

[7/477] S10216956_201106070600
  Processing 2011-06-07 06:00:00
100 - done.
0...10...20...30

In [ ]:
# Montagem da Tabela de Deslocamentos
OUT_DIR = '/home/sialm/neurocompS/TF/v2'
def sistema_para_linha(s):
    """Extrai atributos de um sistema rastreado para um dict."""
    prev = s.relationships[0]

    c_prev = prev.geom.Centroid()
    c_curr = s.geom.Centroid()

    lon_prev, lat_prev = c_prev.GetX(), c_prev.GetY()
    lon_curr, lat_curr = c_curr.GetX(), c_curr.GetY()

    return {
        "name":           str(s.name),
        "timestamp_prev": prev.timestamp,
        "timestamp_curr": s.timestamp,
        "lon_prev":       lon_prev,
        "lat_prev":       lat_prev,
        "lon_curr":       lon_curr,
        "lat_curr":       lat_curr,
        "delta_lon":      lon_curr - lon_prev,
        "delta_lat":      lat_curr - lat_prev,
        "event":          str(s.event),
        "count":          s.attrs.get("count"),
        "area_km2":       s.attrs.get("count") * RESOLUTION**2,
        "tb_min":         s.attrs.get("min"),
        "tb_mean":        s.attrs.get("mean"),
        "tb_std":         s.attrs.get("std"),
        "touching_left":  s.attrs.get("touching_left"),
        "touching_right": s.attrs.get("touching_right"),
        "touching_up":    s.attrs.get("touching_up"),
        "touching_down":  s.attrs.get("touching_down"),
    }


rows = [
    sistema_para_linha(s)
    for s in all_systems_tracked
    if len(s.relationships) > 0
]

df_track = pd.DataFrame(rows)
df_track["timestamp_prev"] = pd.to_datetime(df_track["timestamp_prev"])
df_track["timestamp_curr"] = pd.to_datetime(df_track["timestamp_curr"])
df_track["dt_horas"] = (
    df_track["timestamp_curr"] - df_track["timestamp_prev"]
).dt.total_seconds() / 3600

print("Shape:", df_track.shape)
print("Intervalos (h):", df_track["dt_horas"].value_counts().sort_index().to_dict())
print("Eventos:",        df_track["event"].value_counts().to_dict())
print(df_track.head())

# Salva tabela bruta
df_track.to_csv(os.path.join(OUT_DIR, "tathu_tracking_jun2011.csv"), index=False)
df_track.to_csv(os.path.join(OUT_DIR, "tathu_tracking_jun2011_ponto_virgula.csv"), index=False, sep=";")
print("Tabela bruta salva.")

Shape: (3618, 20)
Intervalos (h): {1.0: 3318, 2.0: 158, 3.0: 142}
Eventos: {'CONTINUITY': 2828, 'SPLIT': 619, 'MERGE': 171}
                                   name timestamp_prev      timestamp_curr  \
0  687743dc-c64c-43b4-b00e-973a5d046b5a     2011-06-07 2011-06-07 01:00:00   
1  8871c9b4-b453-4e76-938b-c8ee66c9919d     2011-06-07 2011-06-07 01:00:00   
2  be120491-aa40-4b61-a69c-483d8323578f     2011-06-07 2011-06-07 01:00:00   
3  678b2b0b-e160-4f65-87d5-639c841a9742     2011-06-07 2011-06-07 01:00:00   
4  53e5f6ea-1df3-4f2a-a2a1-6080eada1edb     2011-06-07 2011-06-07 01:00:00   

    lon_prev  lat_prev   lon_curr  lat_curr  delta_lon  delta_lat       event  \
0 -62.273820  4.885772 -62.498025  4.814061  -0.224205  -0.071711  CONTINUITY   
1 -57.939303  4.150675 -58.180520  3.865692  -0.241217  -0.284983  CONTINUITY   
2 -66.367238 -0.372423 -66.610824 -0.448598  -0.243585  -0.076175  CONTINUITY   
3 -63.152825 -0.695850 -63.454371 -1.147209  -0.301546  -0.451360  CONTINUITY   
4 

In [ ]:
# Filtragem: CONTINUITY + Intervalo de 1h + Sem Borda

df_track_1h = df_track[
    (df_track["dt_horas"] == 1.0) &
    (df_track["event"].str.contains("CONTINUITY")) &
    (~df_track["touching_left"]) &
    (~df_track["touching_right"]) &
    (~df_track["touching_up"]) &
    (~df_track["touching_down"])
].copy()

print(f"Após filtragem: {df_track_1h.shape[0]} linhas  "
      f"(de {df_track.shape[0]} originais)")
print(df_track_1h.head())

df_track_1h.to_csv(os.path.join(OUT_DIR, "tathu_tracking_jun2011_1h.csv"), index=False)
df_track_1h.to_csv(os.path.join(OUT_DIR, "tathu_tracking_jun2011_1h_ponto_virgula.csv"), index=False, sep=";")
print("Tabela filtrada salva.")

Após filtragem: 2186 linhas  (de 3618 originais)
                                   name timestamp_prev      timestamp_curr  \
1  8871c9b4-b453-4e76-938b-c8ee66c9919d     2011-06-07 2011-06-07 01:00:00   
2  be120491-aa40-4b61-a69c-483d8323578f     2011-06-07 2011-06-07 01:00:00   
3  678b2b0b-e160-4f65-87d5-639c841a9742     2011-06-07 2011-06-07 01:00:00   
4  53e5f6ea-1df3-4f2a-a2a1-6080eada1edb     2011-06-07 2011-06-07 01:00:00   
5  178af494-4ba1-4512-9417-432e47989ccc     2011-06-07 2011-06-07 01:00:00   

    lon_prev  lat_prev   lon_curr  lat_curr  delta_lon  delta_lat       event  \
1 -57.939303  4.150675 -58.180520  3.865692  -0.241217  -0.284983  CONTINUITY   
2 -66.367238 -0.372423 -66.610824 -0.448598  -0.243585  -0.076175  CONTINUITY   
3 -63.152825 -0.695850 -63.454371 -1.147209  -0.301546  -0.451360  CONTINUITY   
4 -51.808101 -1.558190 -51.899155 -1.527682  -0.091053   0.030508  CONTINUITY   
5 -55.872798 -1.876015 -56.068922 -2.029962  -0.196124  -0.153947  CONTINUITY

In [ ]:
# pares temporais -1 → t → t+1 e Geração do CSV Final
df_track_1h = df_track_1h.sort_values(["name", "timestamp_curr"]).reset_index(drop=True)

# Cada linha representa t-1 → t.
# Criamos o alvo t → t+1 com um self-join deslocado.
df_next = (
    df_track_1h[["name", "timestamp_prev", "timestamp_curr", "delta_lon", "delta_lat"]]
    .rename(columns={
        "timestamp_prev":  "timestamp_curr",
        "timestamp_curr":  "timestamp_next",
        "delta_lon":       "target_delta_lon",
        "delta_lat":       "target_delta_lat",
    })
)

df_mlp = df_track_1h.merge(df_next, on=["name", "timestamp_curr"], how="inner")

# Garante que o alvo seja exatamente 1 h à frente
dt_target = (df_mlp["timestamp_next"] - df_mlp["timestamp_curr"]).dt.total_seconds() / 3600
df_mlp = df_mlp[dt_target == 1.0].copy()

# Features logarítmicas
df_mlp["log_area_km2"] = np.log1p(df_mlp["area_km2"])
df_mlp["log_count"]    = np.log1p(df_mlp["count"])

print("Dataset MLP:", df_mlp.shape)
print("Período:", df_mlp["timestamp_curr"].min(), "→", df_mlp["timestamp_curr"].max())
print("\nResumo dos alvos:")
print(df_mlp[["target_delta_lon", "target_delta_lat"]].describe())

# Salva CSV final
df_mlp.to_csv(os.path.join(OUT_DIR, "dataset_mlp_regressao_deslocamento.csv"), index=False)
df_mlp.to_csv(os.path.join(OUT_DIR, "dataset_mlp_regressao_deslocamento_ponto_virgula.csv"), index=False, sep=";")
print("Dataset final salvo em:", OUT_DIR)

Dataset MLP: (1097, 25)
Período: 2011-06-07 01:00:00 → 2011-06-29 17:00:00

Resumo dos alvos:
       target_delta_lon  target_delta_lat
count       1097.000000       1097.000000
mean          -0.223927         -0.018103
std            0.192894          0.173326
min           -1.012229         -0.643511
25%           -0.324543         -0.114820
50%           -0.230780         -0.021781
75%           -0.116191          0.078964
max            0.634300          1.184243
Dataset final salvo em: /home/sialm/neurocompS/TF/v2


In [ ]:
# Visualização: Contornos TATHU + Setas MLP

# Pré-requisito: rodar notebook trainMLP para resultar no arquivo com previsao pela MLP



PRED_URL = "https://raw.githubusercontent.com/sialm2020/MLP-no-rasreamento-de-sistemas-convectivos-com-TATHU/main/Results/previsoes_mlp_teste_final.csv"

df_pred_test = pd.read_csv(PRED_URL)

# Converter colunas de tempo
for col in ["timestamp_prev", "timestamp_curr", "timestamp_next"]:
    if col in df_pred_test.columns:
        df_pred_test[col] = pd.to_datetime(df_pred_test[col])

print(df_pred_test.shape)
display(df_pred_test.head())

(255, 38)


,name,timestamp_prev,timestamp_curr,lon_prev,lat_prev,lon_curr,lat_curr,delta_lon,delta_lat,event,...,obs_delta_lon,obs_delta_lat,pred_next_lon,pred_next_lat,obs_next_lon,obs_next_lat,erro_delta_lon,erro_delta_lat,abs_erro_delta_lon,abs_erro_delta_lat
0,0986c625-d8e4-46d4-9f76-bceea7410f4a,2011-06-28 00:00:00,2011-06-28 01:00:00,-60.537536,-3.631513,-60.178843,-4.059393,0.358693,-0.427880,CONTINUITY,...,0.032738,-0.042435,-60.135652,-4.209607,-60.146106,-4.101828,-0.010454,0.107779,0.010454,0.107779
1,0986c625-d8e4-46d4-9f76-bceea7410f4a,2011-06-28 01:00:00,2011-06-28 02:00:00,-60.178843,-4.059393,-60.146106,-4.101828,0.032738,-0.042435,CONTINUITY,...,0.003139,-0.019348,-60.236250,-4.117668,-60.142967,-4.121176,0.093283,-0.003508,0.093283,0.003508
2,0986c625-d8e4-46d4-9f76-bceea7410f4a,2011-06-28 02:00:00,2011-06-28 03:00:00,-60.146106,-4.101828,-60.142967,-4.121176,0.003139,-0.019348,CONTINUITY,...,0.043967,0.080576,-60.264090,-4.110883,-60.099000,-4.040600,0.165090,0.070283,0.165090,0.070283
3,0986c625-d8e4-46d4-9f76-bceea7410f4a,2011-06-28 03:00:00,2011-06-28 04:00:00,-60.142967,-4.121176,-60.099000,-4.040600,0.043967,0.080576,CONTINUITY,...,0.273128,0.077318,-60.249337,-3.964715,-59.825872,-3.963282,0.423465,0.001433,0.423465,0.001433
4,0986c625-d8e4-46d4-9f76-bceea7410f4a,2011-06-28 04:00:00,2011-06-28 05:00:00,-60.099000,-4.040600,-59.825872,-3.963282,0.273128,0.077318,CONTINUITY,...,-0.012167,-0.280135,-59.858956,-3.904292,-59.838039,-4.243416,0.020916,-0.339125,0.020916,0.339125


In [ ]:
# ==========================================================
# FIGURA: TATHU + MLP usando CSV de previsões salvo no GitHub
# ==========================================================

import os
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from matplotlib.lines import Line2D

from tathu.tracking import trackers
from tathu.visualizer import MapView
from tathu.geometry import transform

# ==========================================================
# CONFIGURAÇÕES GERAIS
# ==========================================================

OVERLAP = 0.1      # critério de overlap do tracking
VMIN = 180.0
VMAX = 320.0

# ==========================================================
# CARREGAR CSV DE PREVISÕES
# ==========================================================


df_pred_test["timestamp_prev"] = pd.to_datetime(df_pred_test["timestamp_prev"])
df_pred_test["timestamp_curr"] = pd.to_datetime(df_pred_test["timestamp_curr"])
df_pred_test["timestamp_next"] = pd.to_datetime(df_pred_test["timestamp_next"])

print("CSV carregado:", df_pred_test.shape)
print(df_pred_test.head(3))

# ==========================================================
# FUNÇÕES AUXILIARES
# ==========================================================
def centroid_xy(system):
    """
    Retorna longitude e latitude do centroide de um sistema TATHU.
    """
    c = system.geom.Centroid()
    return c.GetX(), c.GetY()


def distancia_euclidiana(lon1, lat1, lon2, lat2):
    return np.sqrt((lon1 - lon2)**2 + (lat1 - lat2)**2)


def associar_sistemas_previstos(systems_curr, sub, max_dist=0.30):
    """
    Associa cada linha do dataframe 'sub' ao sistema em t mais próximo
    e gera o contorno previsto transladando o sistema atual com base
    em pred_delta_lon e pred_delta_lat.

    Retorna uma lista de dicionários com:
      - row
      - system_curr
      - system_pred
      - dist
    """
    associados = []
    disponiveis = systems_curr.copy()

    for _, row in sub.iterrows():
        if len(disponiveis) == 0:
            break

        melhor_sys = None
        melhor_dist = np.inf
        melhor_idx = None

        for i, sys in enumerate(disponiveis):
            lon_sys, lat_sys = centroid_xy(sys)
            d = distancia_euclidiana(row["lon_curr"], row["lat_curr"], lon_sys, lat_sys)

            if d < melhor_dist:
                melhor_dist = d
                melhor_sys = sys
                melhor_idx = i

        if (melhor_sys is not None) and (melhor_dist <= max_dist):
            sys_pred = copy.deepcopy(melhor_sys)
            sys_pred.geom = transform.translate(
                melhor_sys.geom,
                row["pred_delta_lon"],
                row["pred_delta_lat"]
            )

            associados.append({
                "row": row,
                "system_curr": melhor_sys,
                "system_pred": sys_pred,
                "dist": melhor_dist
            })

            # remove o sistema já associado, para evitar duplicidade
            disponiveis.pop(melhor_idx)

    return associados


def plotar_tathu_mlp_csv(
    ts_plot,
    df_pred_test,
    df_files,
    n_max=6,
    n_labels=4,
    max_dist=0.30,
    salvar=False,
    out_path=None,
    titulo_prefixo="Caso de teste:"
):
    """
    Gera figura TATHU + MLP para um horário específico usando
    previsões salvas em CSV.
    """

    # ------------------------------------------------------
    # Filtrar casos do horário desejado
    # ------------------------------------------------------
    sub = df_pred_test[df_pred_test["timestamp_curr"] == ts_plot].copy()

    if len(sub) == 0:
        raise ValueError(f"Nenhum sistema encontrado em df_pred_test para {ts_plot}.")

    # opcional: escolher os maiores sistemas
    sub = sub.sort_values("area_km2", ascending=False).head(n_max).copy()
    sub = sub.reset_index(drop=True)

    print(f"\nHorário escolhido: {ts_plot}")
    print("Número de linhas do CSV nesse horário:", len(sub))

    # rótulos S1, S2, ...
    sub["rotulo"] = ""
    for i in range(min(n_labels, len(sub))):
        sub.loc[i, "rotulo"] = f"S{i+1}"

    # ------------------------------------------------------
    # Encontrar arquivos de t-1 e t
    # ------------------------------------------------------
    ts_prev = ts_plot - pd.Timedelta(hours=1)

    file_prev = df_files.loc[df_files["timestamp"] == ts_prev, "file"]
    file_curr = df_files.loc[df_files["timestamp"] == ts_plot, "file"]

    if len(file_prev) == 0:
        raise ValueError(f"Arquivo de t-1 não encontrado para {ts_prev}")
    if len(file_curr) == 0:
        raise ValueError(f"Arquivo de t não encontrado para {ts_plot}")

    file_prev = file_prev.iloc[0]
    file_curr = file_curr.iloc[0]

    print("Arquivo t-1:", file_prev)
    print("Arquivo t  :", file_curr)

    # ------------------------------------------------------
    # Rodar detecção e tracking
    # ------------------------------------------------------
    grids = []

    systems_prev = detect(file_prev, grids=grids)
    systems_curr = detect(file_curr, grids=grids)

    grid_prev = grids[0]
    grid_curr = grids[1]

    tracker = trackers.OverlapAreaTracker(
        systems_prev,
        strategy=trackers.RelativeOverlapAreaStrategy(OVERLAP)
    )
    tracker.track(systems_curr)

    # ------------------------------------------------------
    # Criar contornos previstos pela MLP
    # ------------------------------------------------------
    associados = associar_sistemas_previstos(
        systems_curr,
        sub,
        max_dist=max_dist
    )

    systems_pred = [item["system_pred"] for item in associados]

    print("Sistemas em t:", len(systems_curr))
    print("Sistemas destacados do CSV:", len(sub))
    print("Contornos previstos gerados:", len(systems_pred))

    if len(associados) == 0:
        print("Aviso: nenhum sistema foi associado dentro da distância máxima.")

    # ------------------------------------------------------
    # Figura
    # ------------------------------------------------------
    plt.figure(figsize=(12, 8))

    m = MapView(extent)
    m.plotImage(grid_curr, cmap="Greys", vmin=VMIN, vmax=VMAX)

    # contornos t-1
    m.plotSystems(
        systems_prev,
        facecolor="none",
        edgecolor="red",
        centroids=False
    )

    # contornos t
    m.plotSystems(
        systems_curr,
        facecolor="none",
        edgecolor="limegreen",
        centroids=False
    )

    # contornos previstos pela MLP
    if len(systems_pred) > 0:
        m.plotSystems(
            systems_pred,
            facecolor="none",
            edgecolor="orange",
            centroids=False
        )

    ax = plt.gca()

    # ------------------------------------------------------
    # Plotar centroides e rótulos
    # ------------------------------------------------------
    for item in associados:
        row = item["row"]
        sys_curr = item["system_curr"]

        x0, y0 = centroid_xy(sys_curr)

        # centroide atual
        ax.scatter(
            x0, y0,
            s=60,
            c="black",
            marker="o",
            zorder=8
        )

        # rótulo apenas se existir
        if row["rotulo"] != "":
            txt = ax.text(
                x0 + 0.15, y0 + 0.15,
                row["rotulo"],
                fontsize=16,
                weight="bold",
                color="white",
                bbox=dict(facecolor="black", alpha=0.85, pad=2),
                zorder=9
            )
            txt.set_path_effects([pe.withStroke(linewidth=1.5, foreground="black")])

    # ------------------------------------------------------
    # Legenda customizada
    # ------------------------------------------------------
    handles = [
        Line2D([0], [0], color="red", lw=2.5, label="Contorno em t-1"),
        Line2D([0], [0], color="limegreen", lw=2.5, label="Contorno em t"),
        Line2D([0], [0], color="orange", lw=2.5, label="Contorno previsto pela MLP"),
        Line2D([0], [0], marker="o", color="black", markersize=8, linestyle="None",
               label="Centroide atual (t)")
    ]

    ax.legend(
        handles=handles,
        loc="lower left",
        framealpha=0.95,
        facecolor="white",
        edgecolor="black",
        fontsize=12
    )

    plt.title(
        f"{titulo_prefixo} {ts_plot:%d/%m/%Y %H:%M UTC}",
        fontsize=20,
        weight="bold"
    )

    plt.tight_layout()

    if salvar:
        if out_path is None:
            out_path = f"tathu_mlp_{ts_plot:%Y%m%d_%H%M}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        print("Figura salva em:", out_path)

    plt.show()

    return associados



CSV carregado: (255, 38)
                                   name      timestamp_prev  \
0  0986c625-d8e4-46d4-9f76-bceea7410f4a 2011-06-28 00:00:00   
1  0986c625-d8e4-46d4-9f76-bceea7410f4a 2011-06-28 01:00:00   
2  0986c625-d8e4-46d4-9f76-bceea7410f4a 2011-06-28 02:00:00   

       timestamp_curr   lon_prev  lat_prev   lon_curr  lat_curr  delta_lon  \
0 2011-06-28 01:00:00 -60.537536 -3.631513 -60.178843 -4.059393   0.358693   
1 2011-06-28 02:00:00 -60.178843 -4.059393 -60.146106 -4.101828   0.032738   
2 2011-06-28 03:00:00 -60.146106 -4.101828 -60.142967 -4.121176   0.003139   

   delta_lat       event  ...  obs_delta_lon  obs_delta_lat  pred_next_lon  \
0  -0.427880  CONTINUITY  ...       0.032738      -0.042435     -60.135652   
1  -0.042435  CONTINUITY  ...       0.003139      -0.019348     -60.236250   
2  -0.019348  CONTINUITY  ...       0.043967       0.080576     -60.264090   

   pred_next_lat  obs_next_lon  obs_next_lat  erro_delta_lon  erro_delta_lat  \
0      -4.209607

In [ ]:

import matplotlib.patheffects as pe

from matplotlib.lines import Line2D
from tathu.tracking import trackers
from tathu.visualizer import MapView
from tathu.geometry import transform

OVERLAP = 0.1
VMIN = 180.0
VMAX = 320.0


def centroid_xy(system):
    c = system.geom.Centroid()
    return c.GetX(), c.GetY()


def distancia_euclidiana(lon1, lat1, lon2, lat2):
    return np.sqrt((lon1 - lon2)**2 + (lat1 - lat2)**2)


def associar_sistemas_previstos(systems_curr, sub, max_dist=0.30):
    associados = []
    disponiveis = systems_curr.copy()

    for _, row in sub.iterrows():
        if len(disponiveis) == 0:
            break

        melhor_sys = None
        melhor_dist = np.inf
        melhor_idx = None

        for i, sys in enumerate(disponiveis):
            lon_sys, lat_sys = centroid_xy(sys)
            d = distancia_euclidiana(row["lon_curr"], row["lat_curr"], lon_sys, lat_sys)

            if d < melhor_dist:
                melhor_dist = d
                melhor_sys = sys
                melhor_idx = i

        if (melhor_sys is not None) and (melhor_dist <= max_dist):
            sys_pred = copy.deepcopy(melhor_sys)
            sys_pred.geom = transform.translate(
                melhor_sys.geom,
                row["pred_delta_lon"],
                row["pred_delta_lat"]
            )

            associados.append({
                "row": row,
                "system_curr": melhor_sys,
                "system_pred": sys_pred,
                "dist": melhor_dist
            })

            disponiveis.pop(melhor_idx)

    return associados


def plotar_tathu_mlp_csv(
    ts_plot,
    df_pred_test,
    df_files,
    n_max=6,
    n_labels=4,
    max_dist=0.30,
    salvar=False,
    out_path=None,
    mostrar=True,
    titulo_prefixo="Caso de teste:"
):
    sub = df_pred_test[df_pred_test["timestamp_curr"] == ts_plot].copy()

    if len(sub) == 0:
        raise ValueError(f"Nenhum sistema encontrado para {ts_plot}.")

    sub = sub.sort_values("area_km2", ascending=False).head(n_max).copy()
    sub = sub.reset_index(drop=True)

    print(f"\nHorário escolhido: {ts_plot}")
    print("Número de linhas do CSV nesse horário:", len(sub))

    sub["rotulo"] = ""
    for i in range(min(n_labels, len(sub))):
        sub.loc[i, "rotulo"] = f"S{i+1}"

    ts_prev = ts_plot - pd.Timedelta(hours=1)

    file_prev = df_files.loc[df_files["timestamp"] == ts_prev, "file"]
    file_curr = df_files.loc[df_files["timestamp"] == ts_plot, "file"]

    if len(file_prev) == 0:
        raise ValueError(f"Arquivo de t-1 não encontrado para {ts_prev}")
    if len(file_curr) == 0:
        raise ValueError(f"Arquivo de t não encontrado para {ts_plot}")

    file_prev = file_prev.iloc[0]
    file_curr = file_curr.iloc[0]

    print("Arquivo t-1:", file_prev)
    print("Arquivo t  :", file_curr)

    grids = []
    systems_prev = detect(file_prev, grids=grids)
    systems_curr = detect(file_curr, grids=grids)

    grid_prev = grids[0]
    grid_curr = grids[1]

    tracker = trackers.OverlapAreaTracker(
        systems_prev,
        strategy=trackers.RelativeOverlapAreaStrategy(OVERLAP)
    )
    tracker.track(systems_curr)

    associados = associar_sistemas_previstos(
        systems_curr,
        sub,
        max_dist=max_dist
    )

    systems_pred = [item["system_pred"] for item in associados]

    print("Sistemas em t:", len(systems_curr))
    print("Sistemas destacados do CSV:", len(sub))
    print("Contornos previstos gerados:", len(systems_pred))

    plt.figure(figsize=(12, 8))

    m = MapView(extent)
    m.plotImage(grid_curr, cmap="Greys", vmin=VMIN, vmax=VMAX)

    # contornos t-1
    m.plotSystems(
        systems_prev,
        facecolor="none",
        edgecolor="red",
        centroids=False
    )

    # contornos t
    m.plotSystems(
        systems_curr,
        facecolor="none",
        edgecolor="limegreen",
        centroids=False
    )

    # contornos previstos pela MLP
    if len(systems_pred) > 0:
        m.plotSystems(
            systems_pred,
            facecolor="none",
            edgecolor="orange",
            centroids=False
        )

    ax = plt.gca()

    for item in associados:
        row = item["row"]
        sys_curr = item["system_curr"]

        x0, y0 = centroid_xy(sys_curr)

        # centroide atual
        ax.scatter(
            x0, y0,
            s=60,
            c="black",
            marker="o",
            zorder=8
        )

        # rótulos S1, S2...
        if row["rotulo"] != "":
            txt = ax.text(
                x0 + 0.15, y0 + 0.15,
                row["rotulo"],
                fontsize=16,
                weight="bold",
                color="white",
                bbox=dict(facecolor="black", alpha=0.85, pad=2),
                zorder=9
            )
            txt.set_path_effects([pe.withStroke(linewidth=1.5, foreground="black")])

    handles = [
        Line2D([0], [0], color="red", lw=2.5, label="Contorno em t-1"),
        Line2D([0], [0], color="limegreen", lw=2.5, label="Contorno em t"),
        Line2D([0], [0], color="orange", lw=2.5, label="Contorno previsto pela MLP"),
        Line2D([0], [0], marker="o", color="black", markersize=8, linestyle="None",
               label="Centroide atual (t)")
    ]

    ax.legend(
        handles=handles,
        loc="lower left",
        framealpha=0.95,
        facecolor="white",
        edgecolor="black",
        fontsize=12
    )

    plt.title(
        f"{titulo_prefixo} {ts_plot:%d/%m/%Y %H:%M UTC}",
        fontsize=28,
        weight="bold"
    )

    plt.tight_layout()

    if salvar:
        if out_path is None:
            out_path = f"tathu_mlp_{ts_plot:%Y%m%d_%H%M}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        print("Figura salva em:", out_path)

    if mostrar:
        plt.show()
    else:
        plt.close()

    return associados

In [ ]:
def gerar_figuras_teste(
    df_pred_test,
    df_files,
    out_dir,
    n_max=10,
    n_labels=4,
    max_dist=0.30,
    min_sistemas=1,
    titulo_prefixo="Caso de teste:"
):
    """
    Gera automaticamente uma figura para cada horário do conjunto de teste.
    Como df_pred_test já é do teste, usa todos os timestamp_curr presentes nele.
    """

    os.makedirs(out_dir, exist_ok=True)

    # garante datetime
    df_pred_test = df_pred_test.copy()
    df_pred_test["timestamp_curr"] = pd.to_datetime(df_pred_test["timestamp_curr"])

    # conta quantos sistemas existem por horário
    contagem = (
        df_pred_test.groupby("timestamp_curr")
        .size()
        .reset_index(name="n_sistemas")
        .sort_values("timestamp_curr")
    )

    # opcional: filtra horários com pelo menos min_sistemas
    contagem = contagem[contagem["n_sistemas"] >= min_sistemas].copy()

    print("Total de horários a processar:", len(contagem))

    registros = []

    for i, row in enumerate(contagem.itertuples(index=False), start=1):
        ts_plot = row.timestamp_curr
        n_sistemas = row.n_sistemas

        print("\n" + "="*70)
        print(f"[{i}/{len(contagem)}] {ts_plot} | sistemas no CSV: {n_sistemas}")

        nome_arquivo = f"tathu_mlp_teste_{ts_plot:%Y%m%d_%H%M}.png"
        out_path = os.path.join(out_dir, nome_arquivo)

        try:
            associados = plotar_tathu_mlp_csv(
                ts_plot=ts_plot,
                df_pred_test=df_pred_test,
                df_files=df_files,
                n_max=n_max,
                n_labels=n_labels,
                max_dist=max_dist,
                salvar=True,
                out_path=out_path,
                mostrar=False,
                titulo_prefixo=titulo_prefixo
            )

            registros.append({
                "timestamp_curr": ts_plot,
                "n_sistemas_csv": n_sistemas,
                "n_associados_plotados": len(associados),
                "arquivo_figura": out_path,
                "status": "ok"
            })

        except Exception as e:
            print("Erro ao gerar figura:", e)

            registros.append({
                "timestamp_curr": ts_plot,
                "n_sistemas_csv": n_sistemas,
                "n_associados_plotados": np.nan,
                "arquivo_figura": out_path,
                "status": f"erro: {e}"
            })

    df_resumo_figuras = pd.DataFrame(registros)

    resumo_csv = os.path.join(out_dir, "resumo_figuras_teste.csv")
    df_resumo_figuras.to_csv(resumo_csv, index=False)

    print("\nResumo salvo em:", resumo_csv)
    print("Figuras salvas em:", out_dir)

    return df_resumo_figuras

In [ ]:
out_dir = "/home/sialm/neurocompS/TF/figuras_teste_mlp"

df_resumo_figuras = gerar_figuras_teste(
    df_pred_test=df_pred_test,
    df_files=df_files,
    out_dir=out_dir,
    n_max=10,         # máximo de sistemas por figura
    n_labels=4,       # quantos rótulos S1, S2, ...
    max_dist=0.30,    # distância máxima para associação
    min_sistemas=1,   # se quiser, pode colocar 2 ou 3
    titulo_prefixo="Caso de teste:"
)

print(df_resumo_figuras.head())

Total de horários a processar: 81

[1/81] 2011-06-25 01:00:00 | sistemas no CSV: 5

Horário escolhido: 2011-06-25 01:00:00
Número de linhas do CSV nesse horário: 5
Arquivo t-1: /mnt/e/GOES12/jun2011/ch4/06_25/S10216956_201106250000
Arquivo t  : /mnt/e/GOES12/jun2011/ch4/06_25/S10216956_201106250100
  Processing 2011-06-25 00:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 23
  Processing 2011-06-25 01:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90..  Sistemas detectados: 23
Sistemas em t: 23
Sistemas destacados do CSV: 5
Contornos previstos gerados: 5


/home/sialm/miniforge3/envs/env-tathu/lib/python3.13/site-packages/cartopy/mpl/feature_artist.py:144: UserWarning: facecolor will have no effect as it has been defined as "never".
  warnings.warn('facecolor will have no effect as it has been '


Figura salva em: /home/sialm/neurocompS/TF/figuras_teste_mlp/tathu_mlp_teste_20110625_0100.png

[2/81] 2011-06-25 02:00:00 | sistemas no CSV: 4

Horário escolhido: 2011-06-25 02:00:00
Número de linhas do CSV nesse horário: 4
Arquivo t-1: /mnt/e/GOES12/jun2011/ch4/06_25/S10216956_201106250100
Arquivo t  : /mnt/e/GOES12/jun2011/ch4/06_25/S10216956_201106250200
  Processing 2011-06-25 01:00:00
.100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 23
  Processing 2011-06-25 02:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 12
Sistemas em t: 12
Sistemas destacados do CSV: 4
Contornos previstos gerados: 4


/home/sialm/miniforge3/envs/env-tathu/lib/python3.13/site-packages/cartopy/mpl/feature_artist.py:144: UserWarning: facecolor will have no effect as it has been defined as "never".
  warnings.warn('facecolor will have no effect as it has been '


Figura salva em: /home/sialm/neurocompS/TF/figuras_teste_mlp/tathu_mlp_teste_20110625_0200.png

[3/81] 2011-06-25 03:00:00 | sistemas no CSV: 3

Horário escolhido: 2011-06-25 03:00:00
Número de linhas do CSV nesse horário: 3
Arquivo t-1: /mnt/e/GOES12/jun2011/ch4/06_25/S10216956_201106250200
Arquivo t  : /mnt/e/GOES12/jun2011/ch4/06_25/S10216956_201106250300
  Processing 2011-06-25 02:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 12
  Processing 2011-06-25 03:00:00
100 - done.
0...10...20...30...40...50...60...70...80...90...  Sistemas detectados: 8
Sistemas em t: 8
Sistemas destacados do CSV: 3
Contornos previstos gerados: 3


...

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

In [ ]:
#presença de tracker no pacote
import inspect

print("TRACKERS DISPONÍVEIS:")
for name in dir(trackers):
    obj = getattr(trackers, name)
    if inspect.isclass(obj):
        print(name, inspect.signature(obj))

print("\nFORECASTERS DISPONÍVEIS:")
for name in dir(forecasters):
    obj = getattr(forecasters, name)
    if inspect.isclass(obj):
        print(name, inspect.signature(obj))

TRACKERS DISPONÍVEIS:
AbsoluteOverlapAreaStrategy (threshold)
ConvectiveSystemManager (systems)
EdgeTracker (previous)
IntersectsStrategy ()
LifeCycleEvent (*values)
OverlapAreaStrategy ()
OverlapAreaTracker (previous, strategy, picker=<function pick_system_by_max_area at 0x753b50cd6160>)
RelativeOverlapAreaStrategy (threshold)
TitanStrategy (threshold=0.5)

FORECASTERS DISPONÍVEIS:
Conservative (previous, intervals, applyScale=False)
LifeCycleEvent (*values)


In [ ]:
#cria tabela de deslocamento
rows = []

for s in all_systems_tracked:
    # só usa sistemas que têm relação com um sistema anterior
    if len(s.relationships) == 0:
        continue

    prev = s.relationships[0]

    c_prev = prev.geom.Centroid()
    c_curr = s.geom.Centroid()

    lon_prev, lat_prev = c_prev.GetX(), c_prev.GetY()
    lon_curr, lat_curr = c_curr.GetX(), c_curr.GetY()

    rows.append({
        "name": str(s.name),
        "timestamp_prev": prev.timestamp,
        "timestamp_curr": s.timestamp,

        "lon_prev": lon_prev,
        "lat_prev": lat_prev,
        "lon_curr": lon_curr,
        "lat_curr": lat_curr,

        "delta_lon": lon_curr - lon_prev,
        "delta_lat": lat_curr - lat_prev,

        "event": str(s.event),

        "count": s.attrs.get("count"),
        "area_km2": s.attrs.get("count") * resolution**2,
        "tb_min": s.attrs.get("min"),
        "tb_mean": s.attrs.get("mean"),
        "tb_std": s.attrs.get("std"),

        "touching_left": s.attrs.get("touching_left"),
        "touching_right": s.attrs.get("touching_right"),
        "touching_up": s.attrs.get("touching_up"),
        "touching_down": s.attrs.get("touching_down"),
    })

df_track = pd.DataFrame(rows)

df_track["timestamp_prev"] = pd.to_datetime(df_track["timestamp_prev"])
df_track["timestamp_curr"] = pd.to_datetime(df_track["timestamp_curr"])

df_track["dt_horas"] = (
    df_track["timestamp_curr"] - df_track["timestamp_prev"]
).dt.total_seconds() / 3600

print(df_track.head())
print(df_track.shape)
print(df_track["event"].value_counts())
print(df_track["dt_horas"].value_counts().sort_index())

                                   name timestamp_prev      timestamp_curr  \
0  687743dc-c64c-43b4-b00e-973a5d046b5a     2011-06-07 2011-06-07 01:00:00   
1  8871c9b4-b453-4e76-938b-c8ee66c9919d     2011-06-07 2011-06-07 01:00:00   
2  be120491-aa40-4b61-a69c-483d8323578f     2011-06-07 2011-06-07 01:00:00   
3  678b2b0b-e160-4f65-87d5-639c841a9742     2011-06-07 2011-06-07 01:00:00   
4  53e5f6ea-1df3-4f2a-a2a1-6080eada1edb     2011-06-07 2011-06-07 01:00:00   

    lon_prev  lat_prev   lon_curr  lat_curr  delta_lon  delta_lat       event  \
0 -62.273820  4.885772 -62.498025  4.814061  -0.224205  -0.071711  CONTINUITY   
1 -57.939303  4.150675 -58.180520  3.865692  -0.241217  -0.284983  CONTINUITY   
2 -66.367238 -0.372423 -66.610824 -0.448598  -0.243585  -0.076175  CONTINUITY   
3 -63.152825 -0.695850 -63.454371 -1.147209  -0.301546  -0.451360  CONTINUITY   
4 -51.808101 -1.558190 -51.899155 -1.527682  -0.091053   0.030508  CONTINUITY   

   count  area_km2      tb_min     tb_mean  